# Tutorial 03: Building the Grader (Preventing Hallucinations)

In Tutorial 02, we protected our vector database from irrelevant conversational queries using a Semantic Router. However, what happens when the router correctly identifies a technical question, sends it to the vector database, but the database retrieves the wrong documents?

In a Naive RAG system, those irrelevant documents are blindly stuffed into the LLM's prompt. The LLM, trying to be helpful, will often attempt to answer the question using that bad context, resulting in a **hallucination**.

To build a production-grade system, we need a filter between the retrieval step and the generation step. This is the **Document Grader**. Its sole responsibility is to evaluate a `(Question, Document)` pair and output a binary `yes` or `no` indicating relevance.

### Prerequisites
* Completion of Tutorials 01 and 02.
* A local running instance of Ollama.


### Step 1: Defining the Grader

We need an LLM to act as a strict evaluator. Just like the Router, the Grader must return structured JSON so our code can programmatically decide what to do next.


In [ ]:
import json
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate

# 1. Initialize the LLM (using a fast model, temperature=0 for determinism)
llm = Ollama(model="llama3", temperature=0)

# 2. Define the strict grading prompt
grader_template = """You are a strict grader assessing the relevance of a retrieved document to a user question. 
If the document contains keywords or semantic meaning related to the question, grade it as relevant. 

Respond ONLY with a valid JSON object containing a single key 'score' with the value 'yes' or 'no'. Do not provide explanations.

Question: {question}
Document: {document}
JSON Output:"""

grader_prompt = PromptTemplate.from_template(grader_template)

# 3. Build the Grader Chain
grader_chain = grader_prompt | llm

print("✅ Document Grader initialized.")


### Step 2: Testing the Filter

Let's simulate a scenario where our vector database retrieves two documents: one that perfectly answers the user's question, and one completely irrelevant document that slipped in due to a quirk in vector distance.


In [ ]:
# The user's query
query = "How does the system prevent VRAM overflow?"

# Document 1: Highly Relevant
doc_1 = "To prevent VRAM overflow during intensive tasks, the architecture utilizes asymmetric hardware fencing, keeping vector search on the CPU."

# Document 2: Irrelevant (but perhaps retrieved due to the word "system" or "overflow")
doc_2 = "The coffee shop's espresso machine drain system occasionally experiences water overflow if the drip tray is not emptied daily."

def grade_document(question: str, document: str) -> str:
    """Simulates grading a single document."""
    raw_response = grader_chain.invoke({"question": question, "document": document})
    try:
        decision = json.loads(raw_response)
        return decision.get("score", "no")
    except json.JSONDecodeError:
        return "no"

print(f"User Question: {query}\n")

# Grade Document 1
score_1 = grade_document(query, doc_1)
print(f"📄 Document 1: {doc_1}")
print(f"🔍 Grader Decision: {score_1.upper()}\n")

# Grade Document 2
score_2 = grade_document(query, doc_2)
print(f"📄 Document 2: {doc_2}")
print(f"🔍 Grader Decision: {score_2.upper()}")


### The Architectural Win

By passing retrieved documents through this boolean filter, we achieve three critical improvements:

1. **Hallucination Reduction:** The LLM generator never sees `Document 2`, making it impossible for the model to hallucinate a response about "espresso machine drip trays" when asked about GPU VRAM.
2. **Context Window Optimization:** By dropping irrelevant chunks, we save context window space, which lowers token costs and speeds up the final generation step.
3. **Fallback Triggers:** If *all* retrieved documents are graded as "no," the system knows it lacks the context to answer. Instead of guessing, it can trigger a fallback mechanism (like rewriting the user's query to search again).

*Note: In the main repository, this exact logic is encapsulated and batched inside [`scripts/grader.py`](../scripts/grader.py), demonstrating the Single Responsibility Principle.*

---

### 🏋️ Challenge

Currently, our grader only checks if the *document* is relevant to the *question*. But what happens if the LLM generates an answer, and that answer contains made-up facts not found in the document?

**Your Task:**
Write a second prompt template called `hallucination_grader_template`. It should take `document` and `generation` (the LLM's final answer) as inputs, and output a JSON `yes` (if the generation is grounded in the document) or `no` (if the generation contains hallucinations).

In **Tutorial 04: The CRAG State Machine**, we will finally take the Router, the Retriever, the Grader, and the Generator, and wire them together into an autonomous, self-correcting loop using LangGraph.
